# Жизненный цикл сертификата

Ветка: **[ДПО ИИ] 10 — Сертификат или удостоверение**.

Этот ноутбук является частью будущей Jupyter Book-книги и предназначен для воспроизводимой демонстрации модели.

## 1. Состояния документа

Жизненный цикл документа должен поддерживать как стандартную выдачу, так и частичное освоение, отзыв и повторную выдачу.

In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

BASE = Path.cwd()
transitions = [
    {'from':'draft','to':'issued','condition':'все компетенции освоены, итоговый проект принят', 'audit':'certificate.issued'},
    {'from':'draft','to':'partial','condition':'часть компетенций не подтверждена', 'audit':'certificate.partial_recorded'},
    {'from':'partial','to':'issued','condition':'выполнен донабор компетенций', 'audit':'certificate.issued_after_completion'},
    {'from':'issued','to':'revoked','condition':'документ отозван уполномоченным лицом', 'audit':'certificate.revoked'},
    {'from':'issued','to':'reissued','condition':'требуется исправленная версия документа', 'audit':'certificate.reissued'},
    {'from':'reissued','to':'revoked','condition':'перевыпущенный документ также отозван', 'audit':'certificate.revoked'},
]
transitions_df = pd.DataFrame(transitions)
transitions_df

## 2. Диаграмма состояний

Граф отражает допустимые переходы между статусами. Любой критический переход должен фиксироваться в журнале аудита.

In [ ]:
G = nx.DiGraph()
for t in transitions:
    G.add_edge(t['from'], t['to'], label=t['audit'])

plt.figure(figsize=(11, 6))
pos = {
    'draft': (0, 1),
    'partial': (1.8, 0),
    'issued': (3.6, 1),
    'revoked': (5.4, 1.8),
    'reissued': (5.4, 0.2)
}
nx.draw_networkx_nodes(G, pos, node_size=2800)
nx.draw_networkx_labels(G, pos, font_size=10)
nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='-|>', arrowsize=18, connectionstyle='arc3,rad=0.08')
labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=8)
plt.title('Жизненный цикл документа об обучении')
plt.axis('off')
output = BASE / 'assets' / 'diagrams' / 'certificate_lifecycle_graph.png'
plt.savefig(output, dpi=200, bbox_inches='tight')
plt.show()
output

## 3. События аудита

Для научно-методической и инженерной воспроизводимости важно фиксировать не только итоговый статус, но и действия, которые привели к нему.

In [ ]:
audit_events = [
    {'event':'certificate.template.created','actor':'organization_admin','target':'CertificateTemplate','critical':False},
    {'event':'certificate.template.activated','actor':'organization_admin','target':'CertificateTemplate','critical':True},
    {'event':'certificate.generated','actor':'system or admin','target':'IssuedCertificate','critical':True},
    {'event':'certificate.issued','actor':'authorized issuer','target':'IssuedCertificate','critical':True},
    {'event':'certificate.revoked','actor':'organization_admin or super_admin','target':'IssuedCertificate','critical':True},
    {'event':'certificate.reissued','actor':'organization_admin or super_admin','target':'IssuedCertificate','critical':True},
    {'event':'certificate.verified.public','actor':'public','target':'IssuedCertificate','critical':False},
]
pd.DataFrame(audit_events)

## 4. Условия переходов

Переход `draft → issued` допустим только после проверки завершения курса, принятия всех обязательных артефактов и тестов, а также подтверждения итогового проекта. Переходы `issued → revoked` и `issued → reissued` требуют отдельной причины, прав доступа и записи в AuditLog.

In [ ]:
def can_issue(enrollment_status, competencies_mastered, final_project_accepted):
    return enrollment_status == 'completed' and competencies_mastered == 10 and final_project_accepted

examples = pd.DataFrame([
    {'enrollment_status':'completed','competencies_mastered':10,'final_project_accepted':True},
    {'enrollment_status':'completed','competencies_mastered':8,'final_project_accepted':True},
    {'enrollment_status':'active','competencies_mastered':10,'final_project_accepted':True},
])
examples['can_issue'] = examples.apply(lambda r: can_issue(r.enrollment_status, r.competencies_mastered, r.final_project_accepted), axis=1)
examples

## 5. Вывод

Жизненный цикл документа должен быть ограниченным, прозрачным и аудируемым. Это защищает систему от произвольной выдачи документов и позволяет восстановить цепочку решений при спорной ситуации.